In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/modelos-de-fraude/sample_submission.csv
/kaggle/input/competitions/modelos-de-fraude/train.csv
/kaggle/input/competitions/modelos-de-fraude/test.csv


In [2]:
# Cargar archivos de entrenamiento, prueba y plantilla de envío

import pandas as pd
import numpy as np

train_path = "/kaggle/input/competitions/modelos-de-fraude/train.csv"
test_path = "/kaggle/input/competitions/modelos-de-fraude/test.csv"
sample_path = "/kaggle/input/competitions/modelos-de-fraude/sample_submission.csv"

train = pd.read_csv(train_path, sep=";")
test = pd.read_csv(test_path, sep=";")
sample_submission = pd.read_csv(sample_path)

print("Train:", train.shape)
print("Test:", test.shape)
print("Sample submission:", sample_submission.shape)

display(train.head())
display(test.head())
display(sample_submission.head())

Train: (848548, 9)
Test: (200027, 8)
Sample submission: (200027, 2)


,ID,Num_Tarjetas,Balance_Tj,Total_Transac,Trans_Int,Estado,Genero,Tipo_Tarjeta,Fraude
0,1,1,2000,31,9,Alabama,Male,American Express,0
1,2,1,0,25,0,Alabama,Male,American Express,0
2,3,1,2000,78,3,Alabama,Male,American Express,0
3,4,1,2000,11,0,Alabama,Male,American Express,0
4,5,1,2000,40,0,Alabama,Male,American Express,0


,ID,Num_Tarjetas,Balance_Tj,Total_Transac,Trans_Int,Estado,Genero,Tipo_Tarjeta
0,9,1,2000,48,0,Alabama,Male,American Express
1,11,1,2000,5,4,Alabama,Male,American Express
2,19,1,541,21,9,Alabama,Male,American Express
3,20,1,408,44,0,Alabama,Male,American Express
4,22,1,0,5,0,Alabama,Male,American Express


,ID,Fraude
0,9,1
1,11,1
2,19,1
3,20,1
4,22,1


In [3]:
# Revisión básica de la estructura de los datos

print("Columnas train:")
print(train.columns.tolist())

print("\nColumnas test:")
print(test.columns.tolist())

print("\nTipos de datos en train:")
display(train.dtypes)

print("\nValores faltantes en train:")
display(train.isnull().sum())

print("\nValores faltantes en test:")
display(test.isnull().sum())

print("\nDistribución de la variable Fraude:")
display(train["Fraude"].value_counts())

print("\nDistribución porcentual de Fraude:")
display(train["Fraude"].value_counts(normalize=True) * 100)

print("\nDuplicados por ID:")
print("Train:", train["ID"].duplicated().sum())
print("Test:", test["ID"].duplicated().sum())

Columnas train:
['ID', 'Num_Tarjetas', 'Balance_Tj', 'Total_Transac', 'Trans_Int', 'Estado', 'Genero', 'Tipo_Tarjeta', 'Fraude']

Columnas test:
['ID', 'Num_Tarjetas', 'Balance_Tj', 'Total_Transac', 'Trans_Int', 'Estado', 'Genero', 'Tipo_Tarjeta']

Tipos de datos en train:


ID                int64
Num_Tarjetas      int64
Balance_Tj        int64
Total_Transac     int64
Trans_Int         int64
Estado           object
Genero           object
Tipo_Tarjeta     object
Fraude            int64
dtype: object


Valores faltantes en train:


ID               0
Num_Tarjetas     0
Balance_Tj       0
Total_Transac    0
Trans_Int        0
Estado           0
Genero           0
Tipo_Tarjeta     0
Fraude           0
dtype: int64


Valores faltantes en test:


...


Distribución de la variable Fraude:


Fraude
0    797761
1     50787
Name: count, dtype: int64


Distribución porcentual de Fraude:


Fraude
0    94.014835
1     5.985165
Name: proportion, dtype: float64


Duplicados por ID:
Train: 0
Test: 0


In [4]:
# Preparación de variables para la ingeniería de variables

# Copias de trabajo
train_model = train.copy()
test_model = test.copy()

# Ingeniería de variables:
# 1. Proporción de transacciones internacionales sobre el total de transacciones
train_model["Pct_Trans_Int"] = train_model["Trans_Int"] / (train_model["Total_Transac"] + 1)
test_model["Pct_Trans_Int"] = test_model["Trans_Int"] / (test_model["Total_Transac"] + 1)

# 2. Balance promedio por tarjeta
train_model["Balance_Por_Tarjeta"] = train_model["Balance_Tj"] / (train_model["Num_Tarjetas"] + 1)
test_model["Balance_Por_Tarjeta"] = test_model["Balance_Tj"] / (test_model["Num_Tarjetas"] + 1)

# 3. Balance promedio por transacción
train_model["Balance_Por_Transac"] = train_model["Balance_Tj"] / (train_model["Total_Transac"] + 1)
test_model["Balance_Por_Transac"] = test_model["Balance_Tj"] / (test_model["Total_Transac"] + 1)

# 4. Indicador de cliente con transacciones internacionales
train_model["Tiene_Trans_Int"] = (train_model["Trans_Int"] > 0).astype(int)
test_model["Tiene_Trans_Int"] = (test_model["Trans_Int"] > 0).astype(int)

# Separar variable objetivo y variables predictoras
X = train_model.drop(columns=["Fraude"])
y = train_model["Fraude"]

X_test_final = test_model.copy()

print("Variables finales para entrenamiento:")
print(X.columns.tolist())

print("\nTamaño X:", X.shape)
print("Tamaño y:", y.shape)
print("Tamaño X_test_final:", X_test_final.shape)

display(X.head())

Variables finales para entrenamiento:
['ID', 'Num_Tarjetas', 'Balance_Tj', 'Total_Transac', 'Trans_Int', 'Estado', 'Genero', 'Tipo_Tarjeta', 'Pct_Trans_Int', 'Balance_Por_Tarjeta', 'Balance_Por_Transac', 'Tiene_Trans_Int']

Tamaño X: (848548, 12)
Tamaño y: (848548,)
Tamaño X_test_final: (200027, 12)


,ID,Num_Tarjetas,Balance_Tj,Total_Transac,Trans_Int,Estado,Genero,Tipo_Tarjeta,Pct_Trans_Int,Balance_Por_Tarjeta,Balance_Por_Transac,Tiene_Trans_Int
0,1,1,2000,31,9,Alabama,Male,American Express,0.281250,1000.0,62.500000,1
1,2,1,0,25,0,Alabama,Male,American Express,0.000000,0.0,0.000000,0
2,3,1,2000,78,3,Alabama,Male,American Express,0.037975,1000.0,25.316456,1
3,4,1,2000,11,0,Alabama,Male,American Express,0.000000,1000.0,166.666667,0
4,5,1,2000,40,0,Alabama,Male,American Express,0.000000,1000.0,48.780488,0


In [5]:
# Codificación de variables categóricas y separación entrenamiento / validación

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

# Identificación de variables
variables_numericas = [
    "ID",
    "Num_Tarjetas",
    "Balance_Tj",
    "Total_Transac",
    "Trans_Int",
    "Pct_Trans_Int",
    "Balance_Por_Tarjeta",
    "Balance_Por_Transac",
    "Tiene_Trans_Int"
]

variables_categoricas = [
    "Estado",
    "Genero",
    "Tipo_Tarjeta"
]

# Separación de entrenamiento y validación
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Preprocesamiento:
# - Escalamiento para variables numéricas
# - One Hot Encoding para variables categóricas
preprocesador = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), variables_numericas),
        ("cat", OneHotEncoder(handle_unknown="ignore"), variables_categoricas)
    ]
)

print("Tamaño X_train:", X_train.shape)
print("Tamaño X_valid:", X_valid.shape)
print("Tamaño y_train:", y_train.shape)
print("Tamaño y_valid:", y_valid.shape)

print("\nDistribución de Fraude en entrenamiento:")
print(y_train.value_counts(normalize=True) * 100)

print("\nDistribución de Fraude en validación:")
print(y_valid.value_counts(normalize=True) * 100)

Tamaño X_train: (678838, 12)
Tamaño X_valid: (169710, 12)
Tamaño y_train: (678838,)
Tamaño y_valid: (169710,)

Distribución de Fraude en entrenamiento:
Fraude
0    94.014772
1     5.985228
Name: proportion, dtype: float64

Distribución de Fraude en validación:
Fraude
0    94.015085
1     5.984915
Name: proportion, dtype: float64


In [6]:
# Modelo 1: Regresión Logística balanceada

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix

modelo_logistico = Pipeline(
    steps=[
        ("preprocesador", preprocesador),
        ("modelo", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        ))
    ]
)

# Entrenamiento del modelo
modelo_logistico.fit(X_train, y_train)

# Predicción de probabilidades en validación
prob_valid_logistico = modelo_logistico.predict_proba(X_valid)[:, 1]

# Cálculo AUC-ROC
auc_logistico = roc_auc_score(y_valid, prob_valid_logistico)

print("AUC-ROC Regresión Logística:", round(auc_logistico, 6))

AUC-ROC Regresión Logística: 0.937302


In [7]:
# Modelos adicionales: Random Forest y Gradient Boosting

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

resultados_modelos = []

# Guardar resultado del modelo logístico
resultados_modelos.append({
    "Modelo": "Regresión Logística Balanceada",
    "AUC_ROC": auc_logistico
})

# Modelo 2: Random Forest
modelo_rf = Pipeline(
    steps=[
        ("preprocesador", preprocesador),
        ("modelo", RandomForestClassifier(
            n_estimators=150,
            max_depth=10,
            min_samples_leaf=50,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ))
    ]
)

modelo_rf.fit(X_train, y_train)
prob_valid_rf = modelo_rf.predict_proba(X_valid)[:, 1]
auc_rf = roc_auc_score(y_valid, prob_valid_rf)

resultados_modelos.append({
    "Modelo": "Random Forest Balanceado",
    "AUC_ROC": auc_rf
})

print("AUC-ROC Random Forest:", round(auc_rf, 6))


# Modelo 3: Gradient Boosting
modelo_gb = Pipeline(
    steps=[
        ("preprocesador", preprocesador),
        ("modelo", GradientBoostingClassifier(
            n_estimators=150,
            learning_rate=0.05,
            max_depth=3,
            random_state=42
        ))
    ]
)

modelo_gb.fit(X_train, y_train)
prob_valid_gb = modelo_gb.predict_proba(X_valid)[:, 1]
auc_gb = roc_auc_score(y_valid, prob_valid_gb)

resultados_modelos.append({
    "Modelo": "Gradient Boosting",
    "AUC_ROC": auc_gb
})

print("AUC-ROC Gradient Boosting:", round(auc_gb, 6))


# Tabla comparativa
resultados_df = pd.DataFrame(resultados_modelos).sort_values(
    by="AUC_ROC",
    ascending=False
)

display(resultados_df)

AUC-ROC Random Forest: 0.929979
AUC-ROC Gradient Boosting: 0.932123


,Modelo,AUC_ROC
0,Regresión Logística Balanceada,0.937302
2,Gradient Boosting,0.932123
1,Random Forest Balanceado,0.929979


In [8]:
# Entrenamiento final con el mejor modelo y generación del archivo de predicciones

# Se entrena nuevamente el mejor modelo usando toda la base de entrenamiento
modelo_final = Pipeline(
    steps=[
        ("preprocesador", preprocesador),
        ("modelo", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        ))
    ]
)

modelo_final.fit(X, y)

# Predicción de probabilidades sobre la base test
prob_test = modelo_final.predict_proba(X_test_final)[:, 1]

# Crear archivo de envío con la misma estructura de sample_submission
submission = sample_submission.copy()
submission["Fraude"] = prob_test

# Verificar estructura
print("Tamaño submission:", submission.shape)
display(submission.head())

# Guardar archivo CSV para Kaggle
submission.to_csv("submission_logistica_balanceada.csv", index=False)

print("Archivo generado: submission_logistica_balanceada.csv")

Tamaño submission: (200027, 2)


,ID,Fraude
0,9,0.226111
1,11,0.049126
2,19,0.064472
3,20,0.121430
4,22,0.021479


Archivo generado: submission_logistica_balanceada.csv
